In [1]:
%load_ext autoreload
import pandas as pd
from datasets import Dataset
import os
from dotenv import load_dotenv
from TextMiningBasedSatdDetectorModel import TextMiningBasedSatdDetectorModel


/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
DEFAULT_DETECTION_CLASS = 'no'

# Detection Dataset

In [3]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)

detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)


# SATD detector training dataset preparation

In [15]:
BASE_SATD_DETECTOR_DIRECTORY = os.getenv('BASE_SATD_DETECTOR_DIRECTORY')
os.makedirs(BASE_SATD_DETECTOR_DIRECTORY, exist_ok=True)
os.makedirs(os.path.join(BASE_SATD_DETECTOR_DIRECTORY, 'models'), exist_ok=True)
detect_train_df['text'].str.replace('\n', '\t').to_csv(f'{BASE_SATD_DETECTOR_DIRECTORY}/comments.txt', index=False, header=False)
detect_train_df['label'].str.lower().map({'yes': 'Yes', 'no': 'No'}).to_csv(f'{BASE_SATD_DETECTOR_DIRECTORY}/labels.txt', index=False, header=False)
detect_train_df.assign(project='train')['project'].to_csv(f'{BASE_SATD_DETECTOR_DIRECTORY}/projects.txt', index=False, header=False)

In [16]:
pretrained_satd_detector = TextMiningBasedSatdDetectorModel('detect', 'pretrained-liu-detector', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pretrained_satd_detector.fit(detect_train_dataset)
pretrained_satd_detector.predict(detect_test_dataset)

detect with pretrained-liu-detector
../cache/September 11, 2025, 12:58:25$detect_pretrained-liu-detector.csv
Test Result:
              precision    recall  f1-score   support

          no      0.992     0.984     0.988      6855
         yes      0.349     0.536     0.423       112

    accuracy                          0.976      6967
   macro avg      0.671     0.760     0.705      6967
weighted avg      0.982     0.976     0.979      6967



'../cache/September 11, 2025, 12:58:25$detect_pretrained-liu-detector.csv'

In [ ]:
trained_detector = TextMiningBasedSatdDetectorModel('detect', 'trained-liu-detector', {'yes', 'no'}, DEFAULT_DETECTION_CLASS, retrain=True)
trained_detector.fit(detect_train_dataset)
trained_detector.predict(detect_test_dataset)